# Train and validate the physics-informed classifier

The controller remains a NumPy softmax classifier, linear in a nonlinear physics basis. Features compare all throttle levels and encode steering, approach and touchdown phases. Class-balanced training, mirrored demonstrations and a curvature-based optimizer address the failures of the earlier linear model.

All fitting, feature statistics, RBF centres and class weights use training flights only. Validation chooses candidates. Flight success must be checked on the exercise page; this notebook does not run a simulator and cannot certify a successful landing.


In [ ]:
# Reload edits to mymodel.py before each cell.
%load_ext autoreload
%autoreload 2


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mymodel import Controller, OBSERVATION_COLUMNS

landings_path = "landings.csv"
weights_path = "weights.npz"

# L-BFGS iterations include a backtracking line search; these are not SGD epochs.
epochs = 1000
patience = 60
min_delta = 1e-5
critical_clearance = 150.0
critical_descent_speed = 20.0

# Each candidate is a separate experiment. Class weights come from training only.
candidate_settings = {
    "physics_class_weighted": dict(basis="physics", l2=1e-4, class_weight_power=0.5, epochs=350, patience=45),
    "rbf_class_weighted": dict(basis="physics_rbf", n_centres=160, l2=1e-4, class_weight_power=0.5, epochs=350, patience=45),
    "rbf_balanced": dict(basis="physics_rbf", n_centres=160, l2=1e-5, class_weight_power=0.75),
}


In [ ]:
landings = pd.read_csv(landings_path)
print(f"Loaded {len(landings)} observations from {landings_path}")

# Split flights, so observations from a run cannot appear in multiple sets.
run_ids = pd.Series(landings["run_id"].unique()).sample(frac=1, random_state=42).reset_index(drop=True)
n_runs = len(run_ids)
train_end = int(0.8 * n_runs)
val_end = int(0.9 * n_runs)
train_ids, val_ids, test_ids = run_ids[:train_end], run_ids[train_end:val_end], run_ids[val_end:]
train_landings = landings[landings["run_id"].isin(train_ids)].copy()
val_landings = landings[landings["run_id"].isin(val_ids)].copy()
test_landings = landings[landings["run_id"].isin(test_ids)].copy()

for name, subset in [("Training", train_landings), ("Validation", val_landings), ("Test", test_landings)]:
    print(f"{name}: {subset['run_id'].nunique()} flights, {len(subset)} observations")


In [ ]:
numerical_features = list(OBSERVATION_COLUMNS)
display(train_landings.head(5))
display(train_landings[numerical_features].describe())
# Counts and outcomes are diagnostics, never controller inputs.
display(train_landings.groupby("action").size().rename("training_observations"))


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(train_landings["action"].value_counts().index, train_landings["action"].value_counts().values)
plt.title("Distribution of Actions in the Train Landings Dataset")
plt.xlabel("Action")
plt.ylabel("Count")
plt.show()

In [ ]:
# Action numbers are categorical; their numeric correlation is not meaningful.
plt.figure(figsize=(12, 8))
sns.heatmap(train_landings[numerical_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Observation correlations in training flights")
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate action agreement on observations from held-out flights.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def evaluate_actions(controller, data, name):
    y_true = data["action"].to_numpy()
    y_pred = controller.predict(data[numerical_features])
    labels = list(range(10))
    print(f"{name} accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"{name} macro F1 (all 10 actions): {f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0):.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(9, 7))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted action")
    plt.ylabel("Expert action")
    plt.title(f"{name} confusion matrix")
    plt.tight_layout()
    plt.show()

def flat_pad_clearance(data):
    return Controller.ground_clearance(data[numerical_features].to_numpy())

def critical_mask(data):
    return (flat_pad_clearance(data) < critical_clearance) & (data["vy"].to_numpy() < -critical_descent_speed)

def underbraking_stats(data, predictions):
    truth = data["action"].to_numpy()
    # Actions 4-9 are medium or high throttle; 0-3 are off or low.
    needs_burn = critical_mask(data) & (truth >= 4)
    missed = needs_burn & (predictions < 4)
    count = int(needs_burn.sum())
    return count, int(missed.sum()), (float(missed.sum() / count) if count else float("nan"))

def critical_diagnostics(controller, data):
    predictions = controller.predict(data[numerical_features])
    truth = data["action"].to_numpy()
    count, missed, rate = underbraking_stats(data, predictions)
    print(f"Critical observations: {critical_mask(data).sum()}; expert medium/high: {count}; predicted off/low: {missed} ({rate:.1%})")
    print("Action counts (expert and predicted):")
    display(pd.DataFrame({"expert": pd.Series(truth).value_counts(),
                          "predicted": pd.Series(predictions).value_counts()}).reindex(range(10), fill_value=0).fillna(0).astype(int))
    report = pd.DataFrame({"clearance": flat_pad_clearance(data), "descent": -data["vy"].to_numpy(),
                           "expert": truth, "predicted": predictions})
    report["height_band"] = pd.cut(report["clearance"], [-np.inf, 50, 150, 500, np.inf], labels=["0-50", "50-150", "150-500", "500+"])
    report["speed_band"] = pd.cut(report["descent"], [-np.inf, 0, 10, 20, 40, np.inf], labels=["ascending", "0-10", "10-20", "20-40", "40+"])
    report["underbrake"] = (report["expert"] >= 4) & (report["predicted"] < 4)
    report["expert_burn"] = report["expert"] >= 4
    grouped = report.groupby(["height_band", "speed_band"], observed=True).agg(
        observations=("expert", "size"), expert_burns=("expert_burn", "sum"), missed_burns=("underbrake", "sum"))
    grouped["miss_rate"] = grouped["missed_burns"] / grouped["expert_burns"].replace(0, np.nan)
    print("Burn errors by clearance (m) and descent speed (m/s):")
    display(grouped)


In [ ]:
# The test split is not read by this cell. Class weighting is inside Controller.fit.
from threadpoolctl import threadpool_limits
from time import perf_counter

train_X = train_landings[numerical_features].to_numpy()
train_y = train_landings["action"].to_numpy()
val_X = val_landings[numerical_features].to_numpy()
val_y = val_landings["action"].to_numpy()
results, candidates = [], {}

# Limiting BLAS threads avoids oversubscription on small matrix operations.
with threadpool_limits(limits=4):
    for name, settings in candidate_settings.items():
        started = perf_counter()
        print(f"\nTraining {name}: {settings}")
        params = dict(epochs=epochs, patience=patience, min_delta=min_delta)
        params.update(settings)
        candidate = Controller(**params)
        candidate.fit(train_X, train_y, val_X, val_y)
        predictions = candidate.predict(val_X)
        probabilities = candidate.predict_proba(val_X)
        count, missed, miss_rate = underbraking_stats(val_landings, predictions)
        targets = (val_y[:, None] == candidate.classes_).astype(float)
        results.append({"candidate": name,
                        "validation_accuracy": accuracy_score(val_y, predictions),
                        "validation_macro_f1": f1_score(val_y, predictions, labels=range(10), average="macro", zero_division=0),
                        "validation_loss": candidate.cross_entropy_loss(targets, probabilities),
                        "critical_expert_burns": count, "critical_missed_burns": missed,
                        "critical_miss_rate": miss_rate, "features": candidate.weights.shape[0],
                        "best_iteration": candidate.best_epoch_, "iterations": candidate.n_iter_,
                        "seconds": perf_counter()-started})
        candidates[name] = candidate

tuning_results = pd.DataFrame(results).sort_values(
    ["validation_macro_f1", "critical_miss_rate"], ascending=[False, True]).reset_index(drop=True)
# Prefer fewer missed critical burns among models within 0.02 of the best F1.
# The tolerance is an explicit offline selection rule, not a landing guarantee.
minimum_f1 = tuning_results["validation_macro_f1"].max() - 0.02
shortlist = tuning_results[tuning_results["validation_macro_f1"] >= minimum_f1]
selected_name = shortlist.sort_values(["critical_miss_rate", "validation_macro_f1"],
                                      ascending=[True, False]).iloc[0]["candidate"]
controller = candidates[selected_name]
print(f"\nRecommended by validation F1 and critical burn recall: {selected_name}")
print("These offline metrics do not measure flight success. Compare the saved candidates on the exercise page.")
display(tuning_results)


In [ ]:
# weights.npz now always contains the selected candidate, rather than a baseline.
for name, candidate in candidates.items():
    path = f"weights_{name}.npz"
    candidate.save(path)
    print(f"Saved {name}: {path}")
controller.save(weights_path)
controller.save("weights_class_weighted.npz")
tuning_results.to_csv("classifier_tuning.csv", index=False)
print(f"Selected upload: {weights_path} ({selected_name})")
print("Upload this mymodel.py together with the new weights; earlier feature versions are incompatible.")


In [ ]:
plt.figure(figsize=(12, 6))
for name, candidate in candidates.items():
    plt.plot(np.arange(1, len(candidate.validation_loss_history)+1), candidate.validation_loss_history, label=name)
plt.xlabel("L-BFGS iteration")
plt.ylabel("Class-weighted validation cross-entropy")
plt.title("Convergence (loss weights differ between weighting settings)")
plt.legend()
plt.show()


In [ ]:
for name, candidate in candidates.items():
    print(f"\n--- Validation: {name} ---")
    evaluate_actions(candidate, val_landings, f"Validation: {name}")
    critical_diagnostics(candidate, val_landings)

# Sanity checks for the exact same policy used by predict_action on the page.
probe = val_X[:32]
np.testing.assert_array_equal(controller.predict(probe), [controller.predict_action(row) for row in probe])
mirrored = Controller.mirror_observations(probe)
from mymodel import MIRROR_ACTIONS
np.testing.assert_allclose(controller.predict_proba(probe), controller.predict_proba(mirrored)[:, MIRROR_ACTIONS], atol=1e-6)
print("Batch/single prediction and left/right symmetry checks passed.")


### Final held-out test

Choose the candidate using validation and exercise-page flights before inspecting test metrics. The selected candidate is available as `controller`; to select another saved candidate, use for example `controller = candidates["rbf_class_weighted"]`. Save it again if changing the upload selection.

Enable the final test cell once for the selected candidate. Do not tune against its results. New starting states may differ from demonstration flights, and some states may be unrecoverable.


In [ ]:
# Keep disabled during tuning, including Run All.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    evaluate_actions(controller, test_landings, "Final held-out test")
